<a href="https://colab.research.google.com/github/andrewxu319/cuda-fft/blob/main/CUDA_FFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Accidentally reinvented the Stockham FFT after trying to do Cooley-Tukey in-place and without recursion.

In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpfb7w7911".


In [13]:
%%cuda

// to do: ifft currently doesn't work

#include <stdio.h>
#include <cmath>
#include <complex>
#include <vector>
#include <fstream>
#include <cassert>
#include <chrono>
#include <cuda_runtime.h>
#include <cuda/std/complex>

constexpr size_t MAX_BLOCK_SIZE = 1024;
using complex = std::complex<float>;
using cu_complex = cuda::std::complex<float>;
constexpr float pi = 3.14159265358979323846;

void fft_ct_cpu(complex* data, size_t n) {
    if (n == 1) {
        return;
    }

    complex omega{ std::exp(complex{0, - 2 * pi / n}) };
    std::vector<complex> data_e(n/2);
    std::vector<complex> data_o(n/2);
    for (size_t i{}; i < n/2; i++) {
        data_e[i] = data[2 * i];
        data_o[i] = data[2 * i + 1];
    }
    fft_ct_cpu(data_e.data(), n/2);
    fft_ct_cpu(data_o.data(), n/2);

    complex omega_to_i{ 1 };
    for (size_t i{}; i < n/2; i++) {
        data[i] = data_e[i] + omega_to_i * data_o[i];
        data[i + n/2] = data_e[i] - omega_to_i * data_o[i];
        omega_to_i *= omega;
    }
}

void ifft_ct_cpu_recursion(complex* data, size_t n) {
    if (n == 1) {
        return;
    }

    complex omega{ std::exp(complex{0, 2 * pi / n}) };
    std::vector<complex> data_e(n/2);
    std::vector<complex> data_o(n/2);
    for (size_t i{}; i < n/2; i++) {
        data_e[i] = data[2 * i];
        data_o[i] = data[2 * i + 1];
    }
    ifft_ct_cpu_recursion(data_e.data(), n/2);
    ifft_ct_cpu_recursion(data_o.data(), n/2);

    complex omega_to_i{ 1 };
    for (size_t i{}; i < n/2; i++) {
        data[i] = (data_e[i] + omega_to_i * data_o[i]);
        data[i + n/2] = (data_e[i] - omega_to_i * data_o[i]) ;
        omega_to_i *= omega;
    }
}

void ifft_ct_cpu(complex* data, size_t n) {
    ifft_ct_cpu_recursion(data, n);
    float n_reciprocal{ 1.0 / n };
    for (size_t i{}; i < n; i++) {
        data[i] *= n_reciprocal;
    }
}

__device__ void fft_small_kernel_device(cu_complex* full_data_array, cu_complex* data, size_t n, bool ifft) {
    // if large matrix, fft each row
    // here, n means the size of the input for THIS PARTICULAR BLOCK
    size_t i{ threadIdx.x };

    data[i] = full_data_array[blockIdx.x * blockDim.x + i]; // load into shared memory. full_data_array points to the start of the whole input (might exceed 1024), so offset to the start of this particular block
    __syncthreads();

    for (size_t stride{n / 2}; stride > 0; stride /= 2) {
        // note that j resets after i=n/2
        size_t j{ (i % (n/2)) / stride };
        size_t subprob_offset{ i % stride };
        cu_complex y_e_j;
        cu_complex y_o_j;
        y_e_j = data[2 * j * stride + subprob_offset];
        y_o_j = data[(2 * j + 1) * stride + subprob_offset];
        __syncthreads();
        float theta{ 2.0 * pi / (n / stride) * j * (ifft * 2 - 1) };
        float real, imag;
        __sincos(theta, &imag, &real);
        cu_complex omega_to_j{ real, imag };
        data[i] = y_e_j + ((i < n / 2) ? 1 : -1) * omega_to_j * y_o_j;
        __syncthreads();
    }

    full_data_array[blockIdx.x * blockDim.x + i] = data[i];
}

__global__ void fft_small_kernel(cu_complex* data, size_t n, bool ifft) {
    if (ifft) {
        size_t i{ blockDim.x * blockIdx.x + threadIdx.x };
        data[i] /= static_cast<float>(n);
        __syncthreads();
    }
    extern __shared__ cu_complex data_sh[];
    fft_small_kernel_device(data, data_sh, n, ifft);
}

__global__ void fft_small_kernel_called_from_bailey(cu_complex* data, size_t n, bool ifft) {
    extern __shared__ cu_complex data_sh[];
    fft_small_kernel_device(data, data_sh, n, ifft);
}

__device__ void correction_multiplication_kernel(cu_complex* data, size_t n, size_t cols, bool ifft) {
    size_t row{ blockIdx.x };
    size_t col{ threadIdx.x };
    float theta{ 2.0 * pi * row * col / n * (ifft * 2 - 1) };
    float real, imag;
    __sincos(theta, &imag, &real);
    data[row * cols + col] *= cu_complex{ real, imag };
}

__global__ void transpose_kernel(cu_complex* dest, cu_complex* src, size_t n, size_t rows, size_t cols, size_t TILE_DIM, size_t BLOCK_ROWS) {
    // each thread is responsible for several vertically evenly spaced apart entries in the same column
    // based on this https://developer.nvidia.com/blog/efficient-matrix-transpose-cuda-cc/
    extern __shared__ cu_complex tile[]; // original, untransposed

    // location in the whole matrix
    size_t row{ blockIdx.y * TILE_DIM + threadIdx.y };
    size_t col{ blockIdx.x * TILE_DIM + threadIdx.x };

    for (size_t i{}; i < TILE_DIM; i += BLOCK_ROWS) { // i is the index of the start of each of this tile's row relative
        tile[(threadIdx.y + i) * (TILE_DIM + 1) + threadIdx.x] = src[(row + i) * cols + col];
    }

    __syncthreads();

    size_t row_tr{ blockIdx.x * TILE_DIM + threadIdx.y };
    size_t col_tr{ blockIdx.y * TILE_DIM + threadIdx.x };
    for (size_t i{}; i < TILE_DIM; i += BLOCK_ROWS) {
        dest[(row_tr + i) * rows + col_tr] = tile[(threadIdx.x) * (TILE_DIM + 1) + (threadIdx.y + i)];
    }
}

__global__ void bailey_helper_kernel(cu_complex* data, size_t n, size_t rows, size_t cols, bool ifft) {
    size_t row{ blockIdx.x };
    size_t col{ threadIdx.x };
    extern __shared__ cu_complex data_sh[];
    fft_small_kernel_device(data, data_sh, cols, ifft);
    __syncthreads();
    correction_multiplication_kernel(data, n, cols, ifft);
}

__global__ void scaling_kernel(cu_complex* data, float scaling_factor) {
    size_t i{ blockIdx.x * blockDim.x + threadIdx.x };
    data[i] *= scaling_factor;
}

void fft_cuda(cu_complex* h_data, size_t n, bool ifft) {
    cu_complex* d_data;
    cudaMalloc((void**)&d_data, n * sizeof(cu_complex));

    if (n <= MAX_BLOCK_SIZE) {
        cudaMemcpy(d_data, h_data, n * sizeof(cu_complex), cudaMemcpyHostToDevice);;
        fft_small_kernel<<<1, n, n * sizeof(cu_complex)>>>(d_data, n, ifft);
        cudaMemcpy(h_data, d_data, n * sizeof(cu_complex), cudaMemcpyDeviceToHost);
        cudaDeviceSynchronize();
    } else if (n <= MAX_BLOCK_SIZE * MAX_BLOCK_SIZE) { // if possible to fft a whole row & a whole column in one go
        cudaMemcpy(d_data, h_data, n * sizeof(cu_complex), cudaMemcpyHostToDevice);

        // remember n still has to be a power of 2
        size_t cols;
        size_t rows;
        if (n < MAX_BLOCK_SIZE * 32) {
            rows = 32;
            cols = n / 32;
        } else {
            cols = MAX_BLOCK_SIZE;
            rows = n / cols;
        }
        size_t blocks_per_grid{ (n + cols - 1) / cols }; // optimize more

        cu_complex* d_transposed;
        cudaMalloc((void**)&d_transposed, n * sizeof(cu_complex));

        // step 1 fft each row
        // step 2 multiply by correction factor
        bailey_helper_kernel<<<blocks_per_grid, cols, cols * sizeof(cu_complex)>>>(d_data, n, rows, cols, ifft); // each block is a row

        // step 3 transpose
        size_t TILE_DIM{ 32 };
        size_t BLOCK_ROWS{ 8 };
        dim3 grid_dim{ (cols + TILE_DIM - 1) / TILE_DIM, (rows + TILE_DIM - 1) / TILE_DIM, 1 };
        dim3 block_dim{ TILE_DIM, BLOCK_ROWS, 1 }; // CHANGE
        size_t shared_mem_size{ TILE_DIM * (TILE_DIM + 1) * sizeof(cu_complex) }; // +1 for bank conflicts
        transpose_kernel<<<grid_dim, block_dim, shared_mem_size>>>(d_transposed, d_data, n, rows, cols, TILE_DIM, BLOCK_ROWS);

        // step 4 fft each row of transposed matrix
        size_t blocks_per_grid_transposed{ (n + rows - 1) / rows };
        fft_small_kernel_called_from_bailey<<<blocks_per_grid_transposed, rows, rows * sizeof(cu_complex)>>>(d_transposed, rows, ifft);

        // step 5 transpose back
        dim3 grid_dim_transposed{ (rows + TILE_DIM - 1) / TILE_DIM, (cols + TILE_DIM - 1) / TILE_DIM, 1 };
        transpose_kernel<<<grid_dim_transposed, block_dim, shared_mem_size>>>(d_data, d_transposed, n, cols, rows, TILE_DIM, BLOCK_ROWS);

        // step 6 scale by 1/n
        if (ifft) {
            scaling_kernel<<<blocks_per_grid, cols>>>(d_data, 1.0 / n);
        }

        cudaMemcpy(h_data, d_data, n * sizeof(cu_complex), cudaMemcpyDeviceToHost);
        cudaDeviceSynchronize();

        cudaFree(d_transposed);
    } else {
        printf("Size limit\n");
    }

    cudaFree(d_data);
}

int main() {
    // cuda
    std::string line;
    std::ifstream file{ "input.txt" };
    std::vector<cu_complex> data_vector{};
    while (std::getline(file, line)) {
        data_vector.emplace_back(cu_complex{ std::stoi(line), 0.0 });
    }
    size_t n{ data_vector.size() };
    assert(n > 0 && ((n & (n - 1)) == 0)); // power of 2
    cu_complex* data = data_vector.data();

    auto start_time{ std::chrono::high_resolution_clock::now() };
    fft_cuda(data, n, false);
    // fft_cuda(data, n, true);
    auto end_time{ std::chrono::high_resolution_clock::now() };
    int time_elapsed_ms{ std::chrono::duration_cast<std::chrono::milliseconds>(end_time - start_time).count() };

    // // cpu
    // std::string line;
    // std::ifstream file{ "input.txt" };
    // std::vector<complex> data_vector{};
    // while (std::getline(file, line)) {
    //     data_vector.emplace_back(complex{ std::stoi(line), 0.0 });
    // }
    // size_t n{ data_vector.size() };
    // assert(n > 0 && ((n & (n - 1)) == 0)); // power of 2
    // complex* data = data_vector.data();

    // auto start_time{ std::chrono::high_resolution_clock::now() };
    // fft_ct_cpu(data, n);
    // // ifft_ct_cpu(data, n);
    // auto end_time{ std::chrono::high_resolution_clock::now() };
    // int time_elapsed_ms{ std::chrono::duration_cast<std::chrono::milliseconds>(end_time - start_time).count() };

    /////////////

    printf("Time: %d ms\n", time_elapsed_ms);
    for (size_t i{}; i < n; i++) {
        printf("%d, %f", i, std::abs(data[i]));
        // printf("%d, %f", i, data[i].real());
        printf("\n");
    }
    printf("Time: %d ms\n", time_elapsed_ms);
}

Time: 291 ms

